# Reproducibility and Testing in PyTorch

**What you'll learn:**
- How to make PyTorch experiments fully reproducible
- Seeding strategies across all random sources
- Deterministic algorithms and their trade-offs
- Writing proper tests with PyTorch's testing utilities
- Gradient checking and benchmarking

**Prerequisites:** Basic PyTorch training loops, autograd fundamentals

In [ ]:
import torch
import torch.nn as nn
import random
import numpy as np
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"NumPy version: {np.__version__}")

## The Reproducibility Problem

Run this cell multiple times — you'll get different results each time. Neural network training involves multiple sources of randomness: weight initialization, data shuffling, dropout, and (on GPU) non-deterministic algorithms.

In [ ]:
# Demonstration: non-reproducible results
def train_one_step():
    model = nn.Linear(10, 1)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
    x = torch.randn(32, 10)
    y = torch.randn(32, 1)
    loss = nn.functional.mse_loss(model(x), y)
    loss.backward()
    optimizer.step()
    return loss.item()

results = [train_one_step() for _ in range(5)]
print("5 runs, same code, different results:")
for i, r in enumerate(results):
    print(f"  Run {i+1}: loss = {r:.6f}")
print(f"\nMax difference: {max(results) - min(results):.6f}")

## Setting ALL Seeds

To get reproducible results, you must seed **every** source of randomness. PyTorch has its own RNG, Python has `random`, NumPy has its own, and CUDA has per-device generators.

In [ ]:
def seed_everything(seed=42):
    """Seed all sources of randomness for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)  # Seeds CPU and CUDA
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)  # All GPU devices

# Now results are reproducible
seed_everything(42)
result_a = train_one_step()
seed_everything(42)
result_b = train_one_step()

print(f"Run A: {result_a:.10f}")
print(f"Run B: {result_b:.10f}")
print(f"Identical: {result_a == result_b}")

# Different seeds give different (but reproducible) results
seed_everything(123)
result_c = train_one_step()
print(f"\nRun C (seed=123): {result_c:.10f}")
print(f"Different from A: {result_a != result_c}")

## Deterministic Mode

Even with the same seed, some CUDA operations use non-deterministic algorithms (atomicAdd, etc.) for performance. `torch.use_deterministic_algorithms(True)` forces deterministic alternatives.

In [ ]:
# Deterministic mode on CPU
torch.use_deterministic_algorithms(True)

# This makes some operations raise errors if no deterministic implementation exists
print(f"Deterministic algorithms: {torch.are_deterministic_algorithms_enabled()}")

# On CPU, most operations are already deterministic
# The main benefit is on CUDA where some ops use non-deterministic atomics
model_det = nn.Linear(10, 10)
x = torch.randn(5, 10)
out1 = model_det(x)
out2 = model_det(x)
print(f"Same input, same output: {torch.equal(out1, out2)}")

# Reset to default for rest of notebook
torch.use_deterministic_algorithms(False)
print(f"\nDeterministic mode disabled for remaining examples")

## DataLoader Reproducibility

The DataLoader has its own sources of randomness: shuffling order and worker process seeds. To make it reproducible, you need to provide a generator and a worker init function.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

def seed_worker(worker_id):
    """Ensure each DataLoader worker is seeded deterministically."""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# Create a simple dataset
dataset = TensorDataset(torch.randn(100, 5), torch.randint(0, 2, (100,)))

# Reproducible DataLoader
g = torch.Generator()
g.manual_seed(42)

loader = DataLoader(
    dataset,
    batch_size=10,
    shuffle=True,
    worker_init_fn=seed_worker,
    generator=g,
)

# Verify reproducibility
def get_first_batch(seed):
    g = torch.Generator()
    g.manual_seed(seed)
    loader = DataLoader(dataset, batch_size=10, shuffle=True, generator=g)
    return next(iter(loader))[0]

batch_a = get_first_batch(42)
batch_b = get_first_batch(42)
batch_c = get_first_batch(99)

print(f"Same seed → same batch: {torch.equal(batch_a, batch_b)}")
print(f"Diff seed → diff batch: {not torch.equal(batch_a, batch_c)}")

## Writing Tests with PyTorch's TestCase

PyTorch provides a specialized `TestCase` class with tensor-aware assertions. This is the standard way to test PyTorch code — both internally and in extensions.

In [ ]:
import unittest

class TestMyModel(unittest.TestCase):
    def test_output_shape(self):
        """Test that model produces correct output shape."""
        model = nn.Linear(10, 5)
        x = torch.randn(3, 10)
        out = model(x)
        self.assertEqual(out.shape, torch.Size([3, 5]))

    def test_tensor_equality(self):
        """Test approximate tensor equality (handles floating point)."""
        a = torch.tensor([1.0, 2.0, 3.0])
        b = torch.tensor([1.0, 2.0, 3.0]) + 1e-8
        # assertEqual checks approximate equality for floating point
        torch.testing.assert_close(a, b, atol=1e-7, rtol=1e-5)
        print("  Tensors are approximately equal ✓")

    def test_gradient_flow(self):
        """Test that gradients flow through a model."""
        model = nn.Sequential(nn.Linear(10, 5), nn.ReLU(), nn.Linear(5, 1))
        x = torch.randn(2, 10)
        loss = model(x).sum()
        loss.backward()

        for name, p in model.named_parameters():
            self.assertIsNotNone(p.grad, f"No gradient for {name}")
            self.assertFalse(
                torch.all(p.grad == 0),
                f"Zero gradient for {name}"
            )
        print("  All parameters have non-zero gradients ✓")

# Run the tests
suite = unittest.TestLoader().loadTestsFromTestCase(TestMyModel)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)
print()

## Testing Gradients: gradcheck and gradgradcheck

`torch.autograd.gradcheck` verifies that analytical gradients match numerical (finite-difference) gradients. This is the gold standard for testing custom autograd functions.

In [ ]:
# gradcheck: verify analytical vs numerical gradients
from torch.autograd import gradcheck, gradgradcheck

# Custom function to test
class MySoftplus(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        exp_x = torch.exp(x)
        ctx.save_for_backward(exp_x)
        return torch.log(1 + exp_x)

    @staticmethod
    def backward(ctx, grad_output):
        exp_x, = ctx.saved_tensors
        return grad_output * exp_x / (1 + exp_x)

# Test with double precision (gradcheck needs float64 for numerical accuracy)
x = torch.randn(5, 3, dtype=torch.float64, requires_grad=True)

# gradcheck: does the Jacobian match?
passed = gradcheck(MySoftplus.apply, (x,), eps=1e-6, atol=1e-4)
print(f"gradcheck passed: {passed}")

# gradgradcheck: are second derivatives correct too?
passed2 = gradgradcheck(MySoftplus.apply, (x,), eps=1e-6, atol=1e-4)
print(f"gradgradcheck passed: {passed2}")

## Benchmarking Properly

Timing PyTorch code is tricky — you need warmup, synchronization (for CUDA), and statistical rigor. `torch.utils.benchmark.Timer` handles all of this.

In [ ]:
import torch.utils.benchmark as benchmark

# Simple benchmark: matrix multiplication at different sizes
results = []
for n in [64, 128, 256, 512]:
    timer = benchmark.Timer(
        stmt="torch.mm(a, b)",
        globals={"a": torch.randn(n, n), "b": torch.randn(n, n)},
        label="Matrix multiply",
        sub_label=f"{n}x{n}",
        description="torch.mm",
    )
    results.append(timer.blocked_autorange(min_run_time=0.1))

# Display results as a comparison table
compare = benchmark.Compare(results)
compare.print()

### Comparing Two Approaches with Timer

You can benchmark multiple implementations side-by-side to choose the fastest:

In [ ]:
# Compare relu implementations
results = []
for n in [128, 512]:
    x = torch.randn(n, n)

    for impl_name, stmt in [
        ("F.relu", "torch.nn.functional.relu(x)"),
        ("torch.clamp", "torch.clamp(x, min=0)"),
        ("x * (x > 0)", "x * (x > 0).float()"),
    ]:
        timer = benchmark.Timer(
            stmt=stmt,
            globals={"x": x, "torch": torch},
            label="ReLU implementations",
            sub_label=f"{n}x{n}",
            description=impl_name,
        )
        results.append(timer.blocked_autorange(min_run_time=0.1))

compare = benchmark.Compare(results)
compare.print()

## Common Reproducibility Pitfalls

Even with careful seeding, several common mistakes break reproducibility:

In [ ]:
# Pitfall 1: cudnn.benchmark (non-deterministic algorithm selection)
print("Pitfall 1: cudnn.benchmark")
print(f"  torch.backends.cudnn.benchmark = {torch.backends.cudnn.benchmark}")
print("  When True, cuDNN selects fastest algorithm (may vary run-to-run)")
print("  Fix: set to False for reproducibility\n")

# Pitfall 2: Dropout is random — need to set model to eval or seed
print("Pitfall 2: Dropout")
model_with_dropout = nn.Sequential(nn.Linear(10, 10), nn.Dropout(0.5), nn.Linear(10, 1))
x = torch.randn(1, 10)
model_with_dropout.train()
out1 = model_with_dropout(x)
out2 = model_with_dropout(x)
print(f"  Train mode: same input, same output? {torch.equal(out1, out2)}")
model_with_dropout.eval()
out3 = model_with_dropout(x)
out4 = model_with_dropout(x)
print(f"  Eval mode:  same input, same output? {torch.equal(out3, out4)}")

# Pitfall 3: DataLoader with num_workers > 0 and no worker_init_fn
print("\nPitfall 3: DataLoader workers")
print("  Without worker_init_fn, each worker gets its own random state")
print("  Fix: use worker_init_fn as shown earlier\n")

# Pitfall 4: Model weight initialization order matters
print("Pitfall 4: Weight initialization")
torch.manual_seed(42)
m1 = nn.Linear(10, 10)
torch.manual_seed(42)
m2 = nn.Linear(10, 10)
print(f"  Same seed → same weights: {torch.equal(m1.weight, m2.weight)}")
torch.manual_seed(42)
_ = torch.randn(5)  # consume some random numbers
m3 = nn.Linear(10, 10)
print(f"  Same seed but consumed RNG: {torch.equal(m1.weight, m3.weight)}")

## Complete Reproducibility Checklist

Here's a comprehensive function that sets up full reproducibility:

In [ ]:
def make_reproducible(seed=42):
    """Complete reproducibility setup."""
    # 1. Seed all RNGs
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # 2. Deterministic algorithms
    torch.use_deterministic_algorithms(True, warn_only=True)

    # 3. Disable cudnn benchmark
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

    print(f"Reproducibility configured with seed={seed}")
    print(f"  torch.manual_seed: {seed}")
    print(f"  np.random.seed: {seed}")
    print(f"  random.seed: {seed}")
    print(f"  deterministic_algorithms: True (warn_only)")
    print(f"  cudnn.benchmark: False")
    print(f"  cudnn.deterministic: True")

make_reproducible(42)

# Verify it works
def full_experiment(seed):
    make_reproducible(seed)
    model = nn.Sequential(nn.Linear(10, 20), nn.ReLU(), nn.Linear(20, 1))
    optim = torch.optim.SGD(model.parameters(), lr=0.01)
    x = torch.randn(16, 10)
    y = torch.randn(16, 1)
    for _ in range(5):
        loss = nn.functional.mse_loss(model(x), y)
        optim.zero_grad()
        loss.backward()
        optim.step()
    return loss.item()

print(f"\nExperiment A: {full_experiment(42):.10f}")
print(f"Experiment B: {full_experiment(42):.10f}")
print(f"Identical: {full_experiment(42) == full_experiment(42)}")

# Reset for rest of notebook
torch.use_deterministic_algorithms(False)

## 🏋️ Try It Yourself: Write a Test for a Custom Autograd Function

**Exercise:** Implement a custom autograd function for the "swish" activation (x * sigmoid(x)), then write a test that verifies:
1. The forward pass computes the correct value
2. gradcheck passes
3. gradgradcheck passes

In [ ]:
# Exercise starter code

class Swish(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        sigmoid_x = torch.sigmoid(x)
        ctx.save_for_backward(x, sigmoid_x)
        return x * sigmoid_x

    @staticmethod
    def backward(ctx, grad_output):
        x, sigmoid_x = ctx.saved_tensors
        # d/dx [x * sigmoid(x)] = sigmoid(x) + x * sigmoid(x) * (1 - sigmoid(x))
        return grad_output * (sigmoid_x + x * sigmoid_x * (1 - sigmoid_x))

# Your tests here:
# 1. Verify forward matches x * torch.sigmoid(x)
x_test = torch.randn(5, 3, dtype=torch.float64, requires_grad=True)
custom_out = Swish.apply(x_test)
expected_out = x_test * torch.sigmoid(x_test)
print(f"Forward correct: {torch.allclose(custom_out, expected_out)}")

# 2. gradcheck
passed = gradcheck(Swish.apply, (x_test,))
print(f"gradcheck: {passed}")

# 3. gradgradcheck
passed2 = gradgradcheck(Swish.apply, (x_test,))
print(f"gradgradcheck: {passed2}")

print("\nAll tests passed!")

## Key Takeaways

1. **Seed everything** — `random.seed()`, `np.random.seed()`, `torch.manual_seed()`, and `torch.cuda.manual_seed_all()` must all be set

2. **Deterministic mode** — `torch.use_deterministic_algorithms(True)` forces deterministic ops; some operations may error if no deterministic implementation exists

3. **DataLoader needs special care** — provide a `generator` and `worker_init_fn` to make shuffling and worker RNG reproducible

4. **gradcheck/gradgradcheck** — gold standard for testing custom autograd functions; use float64 for numerical accuracy

5. **torch.utils.benchmark.Timer** — proper benchmarking with warmup, synchronization, and statistics; use `Compare` for side-by-side results

6. **Common pitfalls** — `cudnn.benchmark`, dropout in train mode, DataLoader workers, and RNG state consumption can all break reproducibility

7. **warn_only=True** — use `torch.use_deterministic_algorithms(True, warn_only=True)` to get warnings instead of errors for non-deterministic ops

8. **Reproducibility has costs** — deterministic algorithms and disabled cudnn.benchmark can be 10-30% slower